# KIdney dataset

Here we are creating dataset for kidney dataset


In [1]:
import os
from pathlib import Path

current_path = Path.cwd().resolve()
repository_root = next(
    path
    for path in (current_path, *current_path.parents)
    if (path / "pyproject.toml").is_file()
)
os.chdir(repository_root)

repository_root


PosixPath('/home/max/repositories/MSIAutoEncoderWrapper')

In [2]:
# load dataset
from IPython.display import display

from  msi_dataset_manager.exploration import DatasetExplorer

# REMARK: date i download DB is  12.08.2026 (DD, MM, YYYY) 
explorer = DatasetExplorer(
    source="metaspace",
    # Save the METASPACE dataset catalogue in this directory.
    cache_dir="assets/local/datasets/metaspace",
    # Set to True to request the catalogue again and replace the local file.
    refresh_cache=False,
)

# The same interface can be initialized for PRIDE:
# pride_explorer = DatasetExplorer(source="pride")


# About kidney

I choosed kidney as it is most countable dataset


In [3]:
# Replace the key with another enumerable entry from `available_filters`.
organism_values = explorer.get_available_values("organism_part")
display(organism_values.head(30))

# Typical additional inspections:
# display(explorer.get_available_values("organism_part").head(30))
# display(explorer.get_available_values("polarity"))
# display(explorer.get_available_values("analyzer_type").head(30))
# display(explorer.get_available_values("|ionisation_source").head(30))
# display(explorer.get_|available_values("maldi_matrix").head(30))


,value,label,count,variants
0,Kidney,Kidney,4436,"Kidney (4078), kidney (342), Kidney (14), kid..."
1,Brain,Brain,2143,"Brain (2108), brain (35)"
2,Cell Line,Cell Line,971,Cell Line (971)
3,Liver,Liver,945,"Liver (887), liver (51), LIVER (7)"
4,Root,Root,870,"Root (457), root (413)"
5,Lung,Lung,809,"Lung (700), lung (109)"
6,Whole organism,Whole organism,809,"Whole organism (774), whole organism (35)"
7,leaf,leaf,784,"leaf (622), Leaf (162)"
8,Breast,Breast,514,Breast (514)
9,N/A,N/A,425,"N/A (423), n/a (2)"


In [4]:
broad_filters = {
    # Biological metadata
    "organism": "Mouse",
    "organism_part": "Liver",
    "condition": "Wildtype",

    # Acquisition metadata
    "polarity": "Negative",
    # "ionisation_source": "MALDI", - we can normalize this later 

    # Annotation filters
    # "status": "FINISHED",
    "annotation_fdr": 0.1,
    # "has_optical_image": True,
    "min_annotation_count": 1,

}

results = explorer.filter(broad_filters)

display(results)
print(f"Found {len(results)} datasets")


METASPACE discovery:   0%|          | 0/3 [00:00<?, ?stage/s]

Current operation:   0%|          | 0/1 [00:00<?, ?operation/s]

,dataset_id,name,source,project_accession,project_url,organisms,organism_parts,condition,growth_conditions,diseases,...,unannotated_pixel_count,annotated_pixel_fraction,annotation_fdr,spatial_annotation_count,spatial_annotation_database_count,spatial_stats_status,molecule_count,unique_molecule_count,unique_molecules,excluded
0,2026-06-22_15h21m26s,WT_D12_3-neg,metaspace,None,https://metaspace2020.eu/dataset/2026-06-22_15...,Mus musculus (mouse),Liver,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
1,2026-06-22_15h19m59s,WT_D12_2-neg,metaspace,None,https://metaspace2020.eu/dataset/2026-06-22_15...,Mus musculus (mouse),Liver,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
2,2026-06-22_15h19m49s,WT_D12_1-neg,metaspace,None,https://metaspace2020.eu/dataset/2026-06-22_15...,Mus musculus (mouse),Liver,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
3,2026-06-22_15h19m03s,WT_D5_3-neg,metaspace,None,https://metaspace2020.eu/dataset/2026-06-22_15...,Mus musculus (mouse),Liver,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
4,2026-06-22_15h17m46s,WT_D5_2-neg,metaspace,None,https://metaspace2020.eu/dataset/2026-06-22_15...,Mus musculus (mouse),Liver,Wildtype,,,...,None,None,0.1,None,None,None,None,None,,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77,2022-04-14_15h07m57s,2022-04-14_ME_DKFZACLY_S1_W4_DANneg_s10a33_100...,metaspace,None,https://metaspace2020.eu/dataset/2022-04-14_15...,Mus musculus (mouse),Liver,Wildtype,Labeled,,...,None,None,0.1,None,None,None,None,None,,False
78,2022-04-14_14h45m46s,2022-04-13_ME_DKFZACLY_S1_W8_DANneg_s10a33_100...,metaspace,None,https://metaspace2020.eu/dataset/2022-04-14_14...,Mus musculus (mouse),Liver,Wildtype,Unlabeled,,...,None,None,0.1,None,None,None,None,None,,False
79,2017-02-23_09h51m18s,Mouse liver_DMAN_200x200_25um_rcal,metaspace,None,https://metaspace2020.eu/dataset/2017-02-23_09...,Mus musculus (mouse),Liver,Wildtype,N/A,,...,None,None,0.1,None,None,None,None,None,,False
80,2017-02-22_15h01m27s,210217_mouseliver_DMAN_negative_200x200_25um-2...,metaspace,None,https://metaspace2020.eu/dataset/2017-02-22_15...,Mus musculus (mouse),Liver,Wildtype,Caged,,...,None,None,0.1,None,None,None,None,None,,False


Found 82 datasets


Here we can see that proposals that have sense are between 

In [6]:
coverage = explorer.count_mz_range_coverage(
    lower_bounds=[100 * i for i in range(1, 5)],
    upper_bounds=[100 * i for i in range(7, 29)],
)

range_counts_matrix = coverage.pivot(
    index="lower_bound",
    columns="upper_bound",
    values="dataset_count",
)

# from 200 to 900 we obtain 30 datasets, it should be enough
range_counts_matrix

upper_bound,700.0,800.0,900.0,1000.0,1100.0,1200.0,1300.0,1400.0,1500.0,1600.0,...,1900.0,2000.0,2100.0,2200.0,2300.0,2400.0,2500.0,2600.0,2700.0,2800.0
lower_bound,,,,,,,,,,,,,,,,,,,,,
100.0,32,32,32,26,6,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
200.0,51,51,51,44,24,18,18,18,8,0,...,0,0,0,0,0,0,0,0,0,0
300.0,58,58,58,49,29,23,23,23,13,0,...,0,0,0,0,0,0,0,0,0,0
400.0,62,62,62,51,31,25,25,25,13,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
# this one severs for individual filtering
matching = explorer.select_mz_range(
    min_mz=300,
    max_mz=1400,
)
matching

,dataset_id,name,source,project_accession,project_url,organisms,organism_parts,condition,growth_conditions,diseases,...,unannotated_pixel_count,annotated_pixel_fraction,annotation_fdr,spatial_annotation_count,spatial_annotation_database_count,spatial_stats_status,molecule_count,unique_molecule_count,unique_molecules,excluded
0,2025-08-19_19h29m48s,Lipids3_top_control_nodiverter,metaspace,None,https://metaspace2020.eu/dataset/2025-08-19_19...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,319,2,complete,207,5,"C16H14O8-H, C17H18O7-H, C19H32O4-H, C22H21O11-...",False
1,2025-08-19_19h33m37s,Lipids4_top_Nh4F_diverter,metaspace,None,https://metaspace2020.eu/dataset/2025-08-19_19...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,59,2,complete,37,0,,False
2,2025-08-19_19h33m18s,Lipids4_bottom_control_nodiverter,metaspace,None,https://metaspace2020.eu/dataset/2025-08-19_19...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,158,2,complete,109,6,"C12H15NO5-H, C15H10O7S-H, C17H12ClF3N2O-H, C28...",False
3,2025-08-19_19h29m30s,Lipids3_bottom_Nh4F_diverter,metaspace,None,https://metaspace2020.eu/dataset/2025-08-19_19...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,261,2,complete,182,10,"C20H24O3-H, C28H44N2O8S-H, C29H38N5O9-H, C29H4...",False
4,2025-08-19_19h28m58s,Lipids2_top_NH4F_diverter,metaspace,None,https://metaspace2020.eu/dataset/2025-08-19_19...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,348,2,complete,225,12,"C11H20O-H, C13H22O2-H, C14H14N2-H, C14H26O6-H,...",False
5,2025-08-19_19h23m41s,Lipids1_bottom_NH4F_diverter,metaspace,None,https://metaspace2020.eu/dataset/2025-08-19_19...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,417,2,complete,262,44,"C12H15N5O3-H, C17H20N4S-H, C17H22N2O3-H, C18H2...",False
6,2025-08-19_19h27m06s,Lipids2_bottom_control_nodiverter,metaspace,None,https://metaspace2020.eu/dataset/2025-08-19_19...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,180,2,complete,120,1,C33H37N5O5-H,False
7,2025-08-19_19h26m22s,Lipids1_top_control_nodiverter,metaspace,None,https://metaspace2020.eu/dataset/2025-08-19_19...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,309,2,complete,216,9,"C15H20O3-H, C16H26N2O16P2-H, C19H27N5-H, C20H2...",False
8,2025-06-30_15h03m07s,Tissue5_top_NH4F,metaspace,None,https://metaspace2020.eu/dataset/2025-06-30_15...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,1182,2,complete,829,149,"C10H12N2O7-H, C10H12O3-H, C10H13N4O7P-H, C10H1...",False
9,2025-06-30_15h02m48s,Tissue4_top_NH4F,metaspace,None,https://metaspace2020.eu/dataset/2025-06-30_15...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,696,2,complete,468,23,"C10H11NO2-H, C10H18N2O4-H, C11H21N3O5-H, C11H6...",False


In [4]:
filters = {
    # Biological and acquisition metadata
    "organism": "Mouse",
    "organism_part": "Liver",
    "polarity": "Negative",
    "condition": "Wildtype",  # matched tolerantly; also groups "Wildtype", "wildtype", etc.
    "mz_min": 200,
    "mz_max": 1400,

    # Annotation filters and molecular statistics
    "annotation_fdr": 0.1,
    "min_annotation_count": 1,
    "include_molecule_stats": True,
    "include_spatial_annotation_stats": True, # REMARK: it filters unique values based on `annotation_fdr`
}

results_liver = explorer.filter(filters)


METASPACE discovery:   0%|          | 0/3 [00:00<?, ?stage/s]

Current operation:   0%|          | 0/1 [00:00<?, ?operation/s]

In [5]:
display(results_liver)
print(f"Accepted {len(results_liver)} datasets") 

,dataset_id,name,source,project_accession,project_url,organisms,organism_parts,condition,growth_conditions,diseases,...,unannotated_pixel_count,annotated_pixel_fraction,annotation_fdr,spatial_annotation_count,spatial_annotation_database_count,spatial_stats_status,molecule_count,unique_molecule_count,unique_molecules,excluded
0,2025-08-19_19h29m48s,Lipids3_top_control_nodiverter,metaspace,None,https://metaspace2020.eu/dataset/2025-08-19_19...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,319,2,complete,207,5,"C16H14O8-H, C17H18O7-H, C19H32O4-H, C22H21O11-...",False
1,2025-08-19_19h33m37s,Lipids4_top_Nh4F_diverter,metaspace,None,https://metaspace2020.eu/dataset/2025-08-19_19...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,59,2,complete,37,0,,False
2,2025-08-19_19h33m18s,Lipids4_bottom_control_nodiverter,metaspace,None,https://metaspace2020.eu/dataset/2025-08-19_19...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,158,2,complete,109,6,"C12H15NO5-H, C15H10O7S-H, C17H12ClF3N2O-H, C28...",False
3,2025-08-19_19h29m30s,Lipids3_bottom_Nh4F_diverter,metaspace,None,https://metaspace2020.eu/dataset/2025-08-19_19...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,261,2,complete,182,10,"C20H24O3-H, C28H44N2O8S-H, C29H38N5O9-H, C29H4...",False
4,2025-08-19_19h28m58s,Lipids2_top_NH4F_diverter,metaspace,None,https://metaspace2020.eu/dataset/2025-08-19_19...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,348,2,complete,225,12,"C11H20O-H, C13H22O2-H, C14H14N2-H, C14H26O6-H,...",False
5,2025-08-19_19h23m41s,Lipids1_bottom_NH4F_diverter,metaspace,None,https://metaspace2020.eu/dataset/2025-08-19_19...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,417,2,complete,262,44,"C12H15N5O3-H, C17H20N4S-H, C17H22N2O3-H, C18H2...",False
6,2025-08-19_19h27m06s,Lipids2_bottom_control_nodiverter,metaspace,None,https://metaspace2020.eu/dataset/2025-08-19_19...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,180,2,complete,120,1,C33H37N5O5-H,False
7,2025-08-19_19h26m22s,Lipids1_top_control_nodiverter,metaspace,None,https://metaspace2020.eu/dataset/2025-08-19_19...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,309,2,complete,216,9,"C15H20O3-H, C16H26N2O16P2-H, C19H27N5-H, C20H2...",False
8,2025-06-30_15h03m07s,Tissue5_top_NH4F,metaspace,None,https://metaspace2020.eu/dataset/2025-06-30_15...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,1182,2,complete,829,149,"C10H12N2O7-H, C10H12O3-H, C10H13N4O7P-H, C10H1...",False
9,2025-06-30_15h02m48s,Tissue4_top_NH4F,metaspace,None,https://metaspace2020.eu/dataset/2025-06-30_15...,Mouse,Liver,Wildtype,,,...,0,1.0,0.1,696,2,complete,468,23,"C10H11NO2-H, C10H18N2O4-H, C11H21N3O5-H, C11H6...",False


Accepted 18 datasets


In [10]:
## to gigabyte transfer 
results_liver['total_size_bytes'].sum() / 10**(9)

np.float64(1.316907151)

## 4. Export filters


In [11]:
output_path = Path(
    "data/liver_workspace/configs/datasets/liver"
)

exported = explorer.export_selection(
    output_path,
    sort_by="download_size_bytes",
    ascending = False
)

exported

{'filters': PosixPath('data/liver_workspace/configs/datasets/liver/filter.json'),
 'selection': PosixPath('data/liver_workspace/configs/datasets/liver/selection.json')}